In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
from sklearn.utils import check_X_y, check_array
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error


class XGBoostTree:
    """
    Decision tree for XGBoost: uses gradients and Hessians to build splits.
    Implements regularisation (lambda, gamma) and pruning.
    """
    def __init__(self, max_depth=6, min_child_weight=1, lambda_=1.0, gamma=0.0, colsample_bytree=1.0):
        self.max_depth = max_depth
        self.min_child_weight = min_child_weight
        self.lambda_ = lambda_
        self.gamma = gamma
        self.colsample_bytree = colsample_bytree
        self.tree = None   # will be a nested structure of nodes

    class Node:
        __slots__ = ['left', 'right', 'feature', 'threshold', 'weight', 'is_leaf']
        def __init__(self):
            self.left = None
            self.right = None
            self.feature = None
            self.threshold = None
            self.weight = None   # leaf value if leaf, else None
            self.is_leaf = False

    def _compute_gain(self, G, H, G_L, H_L, G_R, H_R):
        """Compute gain for a split given left/right sums."""
        # Gain = 0.5 * (G_L^2/(H_L+λ) + G_R^2/(H_R+λ) - (G_L+G_R)^2/(H_L+H_R+λ)) - γ
        def term(g, h):
            return g**2 / (h + self.lambda_)
        gain = 0.5 * (term(G_L, H_L) + term(G_R, H_R) - term(G_L + G_R, H_L + H_R)) - self.gamma
        return gain

    def _best_split(self, X, G, H, feature_indices):
        """Find the best split among the given features."""
        best_gain = -np.inf
        best_feature = None
        best_threshold = None
        best_left_idx = None
        best_right_idx = None

        n_samples = X.shape[0]
        total_G = np.sum(G)
        total_H = np.sum(H)

        for f in feature_indices:
            # Sort data by feature value
            sorted_idx = np.argsort(X[:, f])
            X_sorted = X[sorted_idx, f]
            G_sorted = G[sorted_idx]
            H_sorted = H[sorted_idx]

            G_L = 0.0
            H_L = 0.0
            # Iterate over possible split points (distinct feature values)
            for i in range(n_samples - 1):
                G_L += G_sorted[i]
                H_L += H_sorted[i]
                G_R = total_G - G_L
                H_R = total_H - H_L

                # Skip if either child has too little weight
                if H_L < self.min_child_weight or H_R < self.min_child_weight:
                    continue

                # Avoid splitting on equal values
                if X_sorted[i] == X_sorted[i + 1]:
                    continue

                gain = self._compute_gain(total_G, total_H, G_L, H_L, G_R, H_R)
                if gain > best_gain:
                    best_gain = gain
                    best_feature = f
                    best_threshold = (X_sorted[i] + X_sorted[i + 1]) / 2.0
                    best_left_idx = sorted_idx[:i+1]
                    best_right_idx = sorted_idx[i+1:]

        return best_feature, best_threshold, best_left_idx, best_right_idx, best_gain

    def _grow_tree(self, X, G, H, depth):
        """Recursively build the tree."""
        node = self.Node()
        total_G = np.sum(G)
        total_H = np.sum(H)
        # Leaf weight = - total_G / (total_H + λ)
        node.weight = -total_G / (total_H + self.lambda_)

        # Check stopping criteria
        if depth >= self.max_depth or len(G) < 2:
            node.is_leaf = True
            return node

        # Select features for this node (column subsampling)
        n_features = X.shape[1]
        if self.colsample_bytree < 1.0:
            n_sub = int(n_features * self.colsample_bytree)
            feature_indices = np.random.choice(n_features, n_sub, replace=False)
        else:
            feature_indices = range(n_features)

        best_feature, best_threshold, left_idx, right_idx, best_gain = self._best_split(
            X, G, H, feature_indices)

        # If no gain or gain <= 0, make leaf
        if best_feature is None or best_gain <= 0:
            node.is_leaf = True
            return node

        node.feature = best_feature
        node.threshold = best_threshold

        # Recurse on children
        X_left, G_left, H_left = X[left_idx], G[left_idx], H[left_idx]
        X_right, G_right, H_right = X[right_idx], G[right_idx], H[right_idx]
        node.left = self._grow_tree(X_left, G_left, H_left, depth + 1)
        node.right = self._grow_tree(X_right, G_right, H_right, depth + 1)

        return node

    def fit(self, X, G, H):
        """Fit the tree to gradients and Hessians."""
        self.tree = self._grow_tree(X, G, H, 0)
        return self

    def _predict_node(self, node, X):
        """Predict for a single node (returns leaf weight)."""
        if node.is_leaf:
            return node.weight * np.ones(X.shape[0])
        else:
            # Split
            left_mask = X[:, node.feature] < node.threshold
            right_mask = ~left_mask
            pred = np.zeros(X.shape[0])
            if np.any(left_mask):
                pred[left_mask] = self._predict_node(node.left, X[left_mask])
            if np.any(right_mask):
                pred[right_mask] = self._predict_node(node.right, X[right_mask])
            return pred

    def predict(self, X):
        """Return tree predictions (leaf weights)."""
        if self.tree is None:
            raise ValueError("Tree not fitted.")
        return self._predict_node(self.tree, X)


class XGBoostBase:
    """Base class for XGBoost with common logic."""
    def __init__(self, n_estimators=100, learning_rate=0.3, max_depth=6,
                 min_child_weight=1, lambda_=1.0, gamma=0.0,
                 colsample_bytree=1.0, random_state=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_child_weight = min_child_weight
        self.lambda_ = lambda_
        self.gamma = gamma
        self.colsample_bytree = colsample_bytree
        self.random_state = random_state
        self.trees = []
        self.initial_pred = None

    def _init_prediction(self, y):
        """Set initial prediction (e.g., mean, log-odds). Overridden in subclasses."""
        raise NotImplementedError

    def _get_gradient_hessian(self, y, pred):
        """Compute gradient and Hessian of loss w.r.t. prediction."""
        raise NotImplementedError

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        np.random.seed(self.random_state)
        self.trees = []
        n_samples = X.shape[0]

        # Initial prediction
        self.initial_pred = self._init_prediction(y)
        pred = np.full(n_samples, self.initial_pred)

        for _ in range(self.n_estimators):
            G, H = self._get_gradient_hessian(y, pred)
            # Fit a tree to gradients and Hessians
            tree = XGBoostTree(
                max_depth=self.max_depth,
                min_child_weight=self.min_child_weight,
                lambda_=self.lambda_,
                gamma=self.gamma,
                colsample_bytree=self.colsample_bytree
            )
            tree.fit(X, G, H)
            # Update prediction
            pred += self.learning_rate * tree.predict(X)
            self.trees.append(tree)

        return self

    def predict_raw(self, X):
        """Return raw ensemble output (log‑odds for classification, value for regression)."""
        X = check_array(X)
        pred = np.full(X.shape[0], self.initial_pred)
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X)
        return pred


class XGBoostRegressor(XGBoostBase, RegressorMixin):
    """XGBoost for regression (squared error loss)."""
    def _init_prediction(self, y):
        return np.mean(y)

    def _get_gradient_hessian(self, y, pred):
        # Loss: 0.5*(y-pred)^2 -> gradient = pred - y, hessian = 1
        G = pred - y
        H = np.ones_like(y)
        return G, H

    def predict(self, X):
        return self.predict_raw(X)


class XGBoostClassifier(XGBoostBase, ClassifierMixin):
    """XGBoost for binary classification (logistic loss)."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.classes_ = None

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-x))

    def _init_prediction(self, y):
        self.classes_ = np.unique(y)
        # Use prior log-odds: log(p/(1-p))
        p_pos = np.mean(y == self.classes_[1])
        p_pos = np.clip(p_pos, 1e-15, 1.0 - 1e-15)
        return np.log(p_pos / (1.0 - p_pos))

    def _get_gradient_hessian(self, y, pred):
        # Loss: -[y log(p) + (1-y) log(1-p)], with p = sigmoid(pred)
        # Gradient = p - y, Hessian = p*(1-p)
        p = self._sigmoid(pred)
        G = p - y   # note: y is 0/1
        H = p * (1.0 - p)
        return G, H

    def predict_proba(self, X):
        raw = self.predict_raw(X)
        p_pos = self._sigmoid(raw)
        p_pos = np.clip(p_pos, 1e-15, 1.0 - 1e-15)
        return np.column_stack((1.0 - p_pos, p_pos))

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.where(proba[:, 1] >= 0.5, self.classes_[1], self.classes_[0])



# Example usage
if __name__ == "__main__":
    # ----- Regression -----
    print("--- Regression ---")
    X, y = make_regression(n_samples=300, n_features=5, noise=10, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    xgb_reg = XGBoostRegressor(n_estimators=50, learning_rate=0.1, max_depth=4,
                               lambda_=0.1, gamma=0.0, random_state=42)
    xgb_reg.fit(X_train, y_train)
    y_pred = xgb_reg.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Test MSE: {mse:.3f}")

    # ----- Binary Classification -----
    print("\n--- Classification ---")
    X, y = make_classification(n_samples=300, n_features=10, n_informative=8,
                               n_redundant=2, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    xgb_clf = XGBoostClassifier(n_estimators=50, learning_rate=0.1, max_depth=4,
                                lambda_=0.1, gamma=0.0, random_state=42)
    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Test accuracy: {acc:.3f}")

    # Show probabilities for first 5 test samples
    proba = xgb_clf.predict_proba(X_test[:5])
    print("Probabilities (first 5):\n", proba)

--- Regression ---
Test MSE: 663.491

--- Classification ---
Test accuracy: 0.878
Probabilities (first 5):
 [[0.80062327 0.19937673]
 [0.75116781 0.24883219]
 [0.56034217 0.43965783]
 [0.94975333 0.05024667]
 [0.86046532 0.13953468]]
